# 🔍 NLP Racism Detection — Training Notebook

Model multi-label untuk deteksi **ujaran kebencian berbasis ras** (HS_Race) dalam teks bahasa Indonesia dan Inggris.

**Label yang dideteksi:**
- `HS` — Hate Speech (ujaran kebencian)
- `HS_Race` — Kebencian berbasis ras / etnis
- `HS_Moderate` / `HS_Strong` — Tingkat keparahan

**Dataset:**
- `re_dataset.csv` — Dataset utama (13K+ tweet Indonesia)
- `racism_dataset_additional.csv` — Dataset tambahan (N-word, ras Indonesia)

## 📦 1. Install & Import Library

In [ ]:
# Install dependencies jika belum ada
# !pip install pandas numpy scikit-learn transformers torch imbalanced-learn matplotlib seaborn tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from tqdm.notebook import tqdm
from collections import Counter

warnings.filterwarnings('ignore')

# Setup device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 📂 2. Load Dataset

In [ ]:
# ── Load dataset utama ──────────────────────────────────────────────────────
df_main = pd.read_csv('re_dataset.csv', encoding='latin-1')
print(f'Dataset utama  : {len(df_main):,} baris')
print(f'Kolom          : {list(df_main.columns)}')
df_main.head(3)

In [ ]:
# ── Load dataset tambahan (racism) ─────────────────────────────────────────
df_add = pd.read_csv('racism_dataset_additional.csv', encoding='utf-8')
print(f'Dataset tambahan: {len(df_add):,} baris')
df_add.head(3)

In [ ]:
# ── Samakan kolom & gabungkan ───────────────────────────────────────────────
LABEL_COLS = ['HS','Abusive','HS_Individual','HS_Group','HS_Religion',
              'HS_Race','HS_Physical','HS_Gender','HS_Other',
              'HS_Weak','HS_Moderate','HS_Strong']

# Tambah kolom Source ke dataset utama jika belum ada
if 'Source' not in df_main.columns:
    df_main['Source'] = 're_dataset'

# Hanya ambil kolom yang diperlukan
keep_cols = ['Tweet'] + LABEL_COLS + ['Source']
df_add_clean = df_add[keep_cols].copy()
df_main_clean = df_main[keep_cols].copy()

# Gabungkan
df = pd.concat([df_main_clean, df_add_clean], ignore_index=True)
print(f'\n✅ Total dataset gabungan: {len(df):,} baris')
print()
print('Distribusi label HS_Race:')
print(df['HS_Race'].value_counts())
print()
print('Distribusi Source:')
print(df['Source'].value_counts())

## 🔎 3. Eksplorasi Data (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribusi Label Dataset', fontsize=16, fontweight='bold')

# Plot distribusi setiap label utama
plot_labels = ['HS', 'HS_Race', 'HS_Moderate', 'HS_Strong']
colors = ['#4F81BD', '#C0504D', '#9BBB59', '#8064A2']

for ax, col, color in zip(axes.flatten(), plot_labels, colors):
    counts = df[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values, color=[color, '#D3D3D3'][:len(counts)])
    ax.set_title(f'Distribusi {col}', fontsize=12)
    ax.set_xlabel('Label')
    ax.set_ylabel('Jumlah')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Plot disimpan ke label_distribution.png')

In [ ]:
# Statistik panjang teks
df['text_len'] = df['Tweet'].astype(str).apply(len)
df['word_count'] = df['Tweet'].astype(str).apply(lambda x: len(x.split()))

print('📏 Statistik panjang teks:')
print(df[['text_len', 'word_count']].describe().round(1))

In [ ]:
# Lihat sampel data HS_Race = 1
print('🔴 Sampel tweet HS_Race = 1 (Rasisme):')
print('─' * 70)
samples = df[df['HS_Race'] == 1]['Tweet'].sample(10, random_state=42)
for i, text in enumerate(samples, 1):
    print(f'{i:2d}. {str(text)[:100]}')

print()
print('🟢 Sampel tweet HS_Race = 0 (Normal):')
print('─' * 70)
samples_neg = df[df['HS_Race'] == 0]['Tweet'].sample(5, random_state=42)
for i, text in enumerate(samples_neg, 1):
    print(f'{i:2d}. {str(text)[:100]}')

## 🧹 4. Preprocessing Teks

In [ ]:
# Kamus normalisasi kata-kata obfuscated (N-word variants, dll.)
OBFUSCATION_MAP = {
    # N-word obfuscations
    r'\bn[1i!][g9][g9][e3]r[s]?\b': 'nigger',
    r'\bn[1i!][g9][g9][a4][s]?\b': 'nigga',
    r'\bni[g9]{2}[a4]\b': 'nigga',
    r'\bn\*+g+\w*\b': 'nigger',
    r'\bn-word\b': 'nigger',
    r'\bthe n word\b': 'nigger',
    # Cina/Tionghoa obfuscations
    r'\bc[1i!]n[o0]\b': 'cino',
    r'\bch[*\-]+nk\b': 'chink',
    r'\bching chong\b': 'chink',
    # Arab
    r'\btowelhead\b': 'arab',
    r'\bsand n[\*\-]+\w+': 'arab',
    # India
    r'\bdot head\b': 'india',
}

def normalize_obfuscation(text):
    """Normalisasi kata-kata yang di-obfuscate."""
    text = str(text).lower()
    for pattern, replacement in OBFUSCATION_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_text(text):
    """Preprocessing teks untuk model."""
    text = str(text)
    # Hapus USER dan URL placeholder
    text = re.sub(r'\bUSER\b', '', text)
    text = re.sub(r'\bURL\b', '', text)
    text = re.sub(r'http\S+', '', text)        # hapus URL
    text = re.sub(r'@\w+', '', text)           # hapus mention
    text = re.sub(r'#\w+', '', text)           # hapus hashtag
    text = re.sub(r'\\n', ' ', text)           # hapus newline literal
    text = re.sub(r'\\x[0-9a-fA-F]{2}', '', text)  # hapus hex escape
    text = re.sub(r'[^\w\s\*\-]', ' ', text)  # hapus karakter non-alfanumerik
    text = re.sub(r'\s+', ' ', text).strip()  # normalkan whitespace
    # Normalisasi obfuscation
    text = normalize_obfuscation(text)
    return text

# Terapkan preprocessing
print('🧹 Preprocessing teks...')
df['Tweet_clean'] = df['Tweet'].apply(clean_text)

# Hapus baris kosong
df = df[df['Tweet_clean'].str.len() > 2].reset_index(drop=True)
print(f'✅ Selesai. Total baris: {len(df):,}')

# Contoh hasil preprocessing
print('\nContoh hasil preprocessing:')
for i in df[df['HS_Race']==1].head(3).itertuples():
    print(f'  Asli   : {str(i.Tweet)[:80]}')
    print(f'  Bersih : {str(i.Tweet_clean)[:80]}')
    print()

## ✂️ 5. Train / Validation / Test Split

In [ ]:
# Target label — fokus pada deteksi rasisme
# Bisa dipilih: 'HS_Race' saja (binary) atau multi-label
TARGET_LABEL = 'HS_Race'   # <── ganti ini untuk target lain

# Pastikan semua label integer
for col in LABEL_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

X = df['Tweet_clean'].values
y = df[TARGET_LABEL].values

# Stratified split: 80% train, 10% val, 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f'📊 Split dataset:')
print(f'  Train      : {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Validation : {len(X_val):,} ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test       : {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)')
print()
print(f'Distribusi label di Train:')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Label {u}: {c:,} ({c/len(y_train)*100:.1f}%)')

## 🤗 6. Tokenizer & Dataset Class (IndoBERT)

In [ ]:
# ─── Pilih model pretrained ────────────────────────────────────────────────
# IndoBERT: terbaik untuk teks Indonesia
# Bisa juga: 'bert-base-multilingual-cased' untuk multilingual
MODEL_NAME = 'indobenchmark/indobert-base-p1'
# MODEL_NAME = 'bert-base-multilingual-cased'   # alternatif
# MODEL_NAME = 'indolem/indobert-base-uncased'  # alternatif lain

MAX_LEN    = 128
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5

print(f'🤖 Model      : {MODEL_NAME}')
print(f'⚙️ Max Length  : {MAX_LEN}')
print(f'⚙️ Batch Size  : {BATCH_SIZE}')
print(f'⚙️ Epochs      : {EPOCHS}')
print(f'⚙️ LR          : {LR}')

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'\n✅ Tokenizer loaded: {MODEL_NAME}')

In [ ]:
class RacismDataset(Dataset):
    """PyTorch Dataset untuk klasifikasi rasisme."""
    
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text  = str(self.texts[idx])
        label = int(self.labels[idx])
        
        encoding = self.tokenizer(
            text,
            max_length        = self.max_len,
            padding           = 'max_length',
            truncation        = True,
            return_tensors    = 'pt',
            return_attention_mask = True
        )
        
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'label'          : torch.tensor(label, dtype=torch.long)
        }

# Buat dataset
train_dataset = RacismDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = RacismDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = RacismDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# Buat DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'✅ Dataset siap:')
print(f'  Train batches : {len(train_loader)}')
print(f'  Val batches   : {len(val_loader)}')
print(f'  Test batches  : {len(test_loader)}')

## 🏗️ 7. Load Model & Setup Training

In [ ]:
# Load model BERT untuk klasifikasi binary (2 kelas: 0=normal, 1=rasisme)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
model = model.to(DEVICE)

# Hitung class weights untuk handle imbalanced dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f'⚖️ Class weights: {dict(zip(np.unique(y_train), class_weights.round(3)))}')

# Loss function dengan class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer dengan weight decay
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# Learning rate scheduler
total_steps    = len(train_loader) * EPOCHS
warmup_steps   = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps
)

print(f'\n✅ Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')
print(f'   Total training steps : {total_steps}')
print(f'   Warmup steps         : {warmup_steps}')

## 🏋️ 8. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    """Satu epoch training."""
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc='  Training', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits  = outputs.logits

        loss = criterion(logits, labels)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


def evaluate(model, loader, criterion, device):
    """Evaluasi model pada val/test set."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='  Evaluating', leave=False):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits

            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1, all_labels, all_preds, all_probs


print('✅ Fungsi training dan evaluasi siap.')

In [ ]:
# ─── Training Loop Utama ──────────────────────────────────────────────────
best_val_f1   = 0.0
best_model_path = 'best_racism_model.pt'
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

print(f'🚀 Mulai training — {EPOCHS} epochs\n')
print('=' * 55)

for epoch in range(1, EPOCHS + 1):
    print(f'\n📅 Epoch {epoch}/{EPOCHS}')

    train_loss, train_f1 = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, DEVICE
    )
    val_loss, val_f1, _, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)

    print(f'  Train → Loss: {train_loss:.4f} | F1: {train_f1:.4f}')
    print(f'  Val   → Loss: {val_loss:.4f} | F1: {val_f1:.4f}', end='')

    # Simpan model terbaik
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  ✅ Model terbaik disimpan!')
    else:
        print()

print(f'\n🏆 Training selesai! Best Val F1: {best_val_f1:.4f}')

## 📈 9. Visualisasi Hasil Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1 Score
axes[1].plot(epochs_range, history['train_f1'], 'b-o', label='Train F1', linewidth=2)
axes[1].plot(epochs_range, history['val_f1'],   'r-o', label='Val F1',   linewidth=2)
axes[1].set_title('Training & Validation F1-Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'Training History — {MODEL_NAME}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training history disimpan ke training_history.png')

## 🎯 10. Evaluasi Final pada Test Set

In [ ]:
# Load model terbaik
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
print(f'✅ Model terbaik di-load dari: {best_model_path}')

# Evaluasi pada test set
test_loss, test_f1, y_true, y_pred, y_probs = evaluate(
    model, test_loader, criterion, DEVICE
)

print(f'\n🎯 Hasil Evaluasi Test Set:')
print(f'  Test Loss : {test_loss:.4f}')
print(f'  Test F1   : {test_f1:.4f}')

print(f'\n📋 Classification Report:')
print('─' * 55)
print(classification_report(
    y_true, y_pred,
    target_names=['Non-Rasisme (0)', 'Rasisme (1)'],
    digits=4
))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 6))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-Rasisme', 'Rasisme'],
    yticklabels=['Non-Rasisme', 'Rasisme'],
    ax=ax, linewidths=0.5, annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Confusion matrix disimpan ke confusion_matrix.png')

In [ ]:
# Metrik tambahan
from sklearn.metrics import roc_curve, auc as sklearn_auc

fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = sklearn_auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Racism Detector', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 ROC AUC = {roc_auc:.4f}')

## 🔬 11. Error Analysis

In [ ]:
# Buat DataFrame hasil prediksi test set
test_texts = X_test
results_df = pd.DataFrame({
    'text'      : test_texts,
    'true_label': y_true,
    'pred_label': y_pred,
    'prob_race' : np.round(y_probs, 4)
})

# False Positives (model salah prediksi rasisme padahal normal)
fp = results_df[(results_df['true_label'] == 0) & (results_df['pred_label'] == 1)]
print(f'🔴 False Positives: {len(fp)}')
print('Teks yang salah diklasifikasi sebagai rasisme:')
for _, row in fp.head(5).iterrows():
    print(f'  [{row["prob_race"]:.3f}] {str(row["text"])[:100]}')

print()

# False Negatives (model gagal deteksi rasisme)
fn = results_df[(results_df['true_label'] == 1) & (results_df['pred_label'] == 0)]
print(f'🟡 False Negatives: {len(fn)}')
print('Teks rasisme yang tidak terdeteksi:')
for _, row in fn.head(5).iterrows():
    print(f'  [{row["prob_race"]:.3f}] {str(row["text"])[:100]}')

# Simpan hasil analisis
results_df.to_csv('test_predictions.csv', index=False, encoding='utf-8-sig')
print(f'\n📁 Hasil prediksi disimpan ke test_predictions.csv')

## 💾 12. Simpan Model & Tokenizer

In [ ]:
# Simpan model dan tokenizer dalam format HuggingFace
MODEL_SAVE_DIR = 'saved_racism_model'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

model.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(MODEL_SAVE_DIR)

# Simpan juga config tambahan
import json
model_config = {
    'model_name'   : MODEL_NAME,
    'target_label' : TARGET_LABEL,
    'max_len'      : MAX_LEN,
    'num_labels'   : 2,
    'label_map'    : {'0': 'Non-Rasisme', '1': 'Rasisme'},
    'best_val_f1'  : round(best_val_f1, 4),
    'test_f1'      : round(test_f1, 4),
    'dataset_size' : len(df),
    'train_size'   : len(X_train),
    'val_size'     : len(X_val),
    'test_size'    : len(X_test)
}
with open(f'{MODEL_SAVE_DIR}/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)

print(f'✅ Model disimpan ke: ./{MODEL_SAVE_DIR}/')
print(json.dumps(model_config, indent=2))

## 🎮 13. Inferensi — Prediksi Teks Baru

In [ ]:
def predict_racism(text, model, tokenizer, device, max_len=128):
    """
    Prediksi apakah teks mengandung rasisme.
    
    Returns:
        dict: {
            'label': 'Rasisme' atau 'Non-Rasisme',
            'confidence': probabilitas (0-1),
            'is_racist': True/False
        }
    """
    model.eval()
    
    # Preprocessing
    clean = clean_text(text)
    
    encoding = tokenizer(
        clean,
        max_length    = max_len,
        padding       = 'max_length',
        truncation    = True,
        return_tensors = 'pt'
    )
    
    with torch.no_grad():
        input_ids      = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()
    
    pred_class = int(np.argmax(probs))
    confidence = float(probs[pred_class])
    
    return {
        'text'       : text[:80] + '...' if len(text) > 80 else text,
        'clean_text' : clean[:80],
        'label'      : 'Rasisme' if pred_class == 1 else 'Non-Rasisme',
        'confidence' : round(confidence, 4),
        'prob_racist': round(float(probs[1]), 4),
        'is_racist'  : pred_class == 1
    }


# ── Contoh Prediksi ──────────────────────────────────────────────────────────
test_examples = [
    # Positif — Rasisme
    "dasar cina pelit gak mau berbagi sama pribumi",
    "nigger get out of here you dont belong",
    "orang negro itu memang biadab dari sananya",
    "usir semua cina dari indonesia sekarang juga",
    "n1gg4 always causing trouble",
    "sipit maju melulu di negeri orang",
    # Negatif — Normal
    "keberagaman etnis di indonesia adalah kekayaan yang harus dijaga",
    "teman saya dari papua sangat cerdas dan berprestasi",
    "we should embrace people from all ethnic backgrounds",
    "saya bangga indonesia memiliki banyak suku dan budaya",
]

print('🎮 Demo Prediksi Rasisme')
print('=' * 65)
for text in test_examples:
    result = predict_racism(text, model, tokenizer, DEVICE)
    icon   = '🔴' if result['is_racist'] else '🟢'
    print(f"{icon} [{result['label']:12s}] (conf: {result['confidence']:.3f}) {result['text']}")

## 📊 14. Ringkasan Hasil

In [ ]:
print('='*65)
print('📊 RINGKASAN TRAINING — NLP RACISM DETECTOR')
print('='*65)
print(f'  Model             : {MODEL_NAME}')
print(f'  Target Label      : {TARGET_LABEL}')
print(f'  Total Dataset     : {len(df):,} baris')
print(f'    - Dataset Utama : {len(df_main):,} (re_dataset.csv)')
print(f'    - Dataset Tambahan: {len(df_add):,} (racism_dataset_additional.csv)')
print(f'  Train/Val/Test    : {len(X_train):,} / {len(X_val):,} / {len(X_test):,}')
print(f'  Epochs            : {EPOCHS}')
print(f'  Batch Size        : {BATCH_SIZE}')
print(f'  Max Sequence Len  : {MAX_LEN}')
print(f'  Learning Rate     : {LR}')
print()
print(f'  Best Val F1       : {best_val_f1:.4f}')
print(f'  Test F1 (Macro)   : {test_f1:.4f}')
print()
print(f'  Model tersimpan   : ./{MODEL_SAVE_DIR}/')
print(f'  Checkpoint        : ./{best_model_path}')
print('='*65)